In [45]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.path import Path
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc

import numpy as np
import os
import plotly.io as pio
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy.sparse as sp
from anndata import AnnData

In [46]:
class PolygonAnnotator:
    def __init__(self, image, cmap='hot'):
        self.image = image
        self.coords = None
        self.done = False

        self.fig, self.ax = plt.subplots()
        self.ax.imshow(image, cmap=cmap)
        self.ax.set_title("Click to draw polygon. Double-click to finish.")

        self.selector = PolygonSelector(
            self.ax,
            self.onselect,
            useblit=True,
            props=dict(color='cyan', linewidth=2),
            handle_props=dict(marker='o', markersize=5, mec='cyan', mfc='cyan')
        )

        self.fig.canvas.mpl_connect('close_event', self._on_close)
        plt.show(block=True)  # Blocks until the window is closed

    def _on_close(self, event):
        self.done = True

    def onselect(self, verts):
        self.coords = verts
        print(f"Polygon drawn with {len(verts)} points.")
        plt.close(self.fig)  # Automatically close the window

    def get_mask(self):
        if not self.coords:
            print("No polygon was drawn.")
            return None
        ny, nx = self.image.shape
        y, x = np.mgrid[:ny, :nx]
        points = np.vstack((x.flatten(), y.flatten())).T
        path = Path(self.coords)
        return path.contains_points(points).reshape((ny, nx))

In [47]:
def get_mz_heatmap_image(adata, target_mz, tol=0.1):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    x = adata.obs["x"].values.astype(int)
    y = adata.obs["y"].values.astype(int)

    width = x.max() + 1
    height = y.max() + 1
    image = np.zeros((height, width))
    image[y, x] = intensities

    return image, matched_mz

In [48]:
# Load and visualize the m/z heatmap from AnnData
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad


In [49]:
def sort_adata_by_mz(adata, mz_column="mzs"):
    def to_float_with_warning(index, label="var_names"):
        mzs = pd.to_numeric(index, errors="coerce")
        n_nans = mzs.isna().sum()
        if n_nans > 0:
            print(f"⚠️ Warning: {n_nans} entries in {label} could not be converted to float and were set to NaN.")
        return mzs
    adata.var[mz_column] = to_float_with_warning(adata.var_names)
    adata = adata[:, adata.var[mz_column].sort_values().index]
    return adata


In [50]:
adata = sort_adata_by_mz(adata, mz_column="mzs")
adata.var

,mzs
136.0150,136.0150
136.0608,136.0608
137.0251,137.0251
138.0287,138.0287
139.0359,139.0359
...,...
1520.1588,1520.1588
1521.1633,1521.1633
1522.1682,1522.1682
1542.1475,1542.1475


In [57]:
# ---- Run ----

target_mz = 788.5

image, matched_mz = get_mz_heatmap_image(adata, target_mz)
print(f"Target m/z: {target_mz} → Matched m/z: {matched_mz}")

custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),   # white
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])

#custom_cmap = "binary"
# Use the annotator with your image and colormap
annotator = PolygonAnnotator(image, cmap=custom_cmap)
mask = annotator.get_mask()

if mask is not None:
    masked_image = np.where(mask, image, 0)
    plt.imshow(masked_image, cmap=custom_cmap)
    plt.title(f"Masked m/z = {matched_mz:.4f}")
    plt.show()
else:
    print("No mask created.")

Target m/z: 788.5 → Matched m/z: 788.5027
No polygon was drawn.
No mask created.


In [ ]:
# Get spatial coordinates
x = adata.obs["x"].astype(int).values
y = adata.obs["y"].astype(int).values

# Mask value at each spot
adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)
adata.obs["tissue"] = adata.obs["tissue"].astype(int)  # 1 = inside, 0 = outside
print(adata.obs["tissue"].value_counts())

tissue
0    9574
1    7031
Name: count, dtype: int64


/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_7801/1596079847.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)


In [ ]:
adata1 = adata
adata.obs["tissue"]

0        0
1        0
2        0
3        0
4        0
        ..
16600    0
16601    0
16602    0
16603    0
16604    0
Name: tissue, Length: 16605, dtype: int64

In [ ]:
adata = adata1
adata

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [ ]:
import numpy as np
import scipy.sparse as sp
import anndata as ad

def tic_normalize(adata, inplace = True, layer_name = None):
    """
    Perform Total Ion Current (TIC) normalization on an AnnData object.
    
    Parameters:
    - adata (AnnData): The input AnnData object with intensity matrix in `.X`.
    - inplace (bool): If True, overwrite `adata.X`. If False, store result in `adata.layers[layer_name]`.
    - layer_name (str): Name of the layer to store normalized matrix. Required if `inplace=False`.

    Returns:
    - None. The AnnData object is modified in-place.
    """

    if not inplace and not layer_name:
        raise ValueError("You must specify 'layer_name' when inplace=False.")

    # Compute TIC per observation
    tic = adata.X.sum(axis=1)
    tic = np.asarray(tic).flatten()
    tic[tic == 0] = 1.0  # Avoid division by zero

    # Normalize
    if sp.issparse(adata.X):
        normalized = adata.X.multiply(1 / tic[:, np.newaxis])
    else:
        normalized = adata.X / tic[:, np.newaxis]

    # Store result
    if inplace:
        adata.layers["raw"] = adata.X.copy()  # Store raw values before normalization
        adata.X = normalized
    else:
        adata.layers[layer_name] = normalized

    # Save TIC to .obs
    adata.obs["tic"] = tic

    print(f"✅ TIC normalization complete. {'Updated adata.X' if inplace else f'Stored in layer: {layer_name}'}")
    

In [ ]:
tic_normalize(adata, inplace=True)

✅ TIC normalization complete. Updated adata.X


In [ ]:
def filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=2.0, layer=None, normalize_tic=True):
    # 1. Extract raw data
    data = adata.layers[layer] if layer is not None else adata.X

    # 2. TIC normalization per pixel
    if normalize_tic:
        # Sum across features for each observation (axis=1)
        tic = np.asarray(data.sum(axis=1)).ravel()
        # Avoid division by zero
        tic[tic == 0] = 1.0
        # Normalize in place (if sparse, convert to array)
        data = data / tic[:, None]

    # 3. Define tissue masks
    mask_tissue = adata.obs[tissue_key] == tissue_val
    mask_non    = adata.obs[tissue_key] == non_tissue_val

    # 4. Compute mean normalized intensity per feature
    mean_tissue    = np.asarray(data[mask_tissue].mean(axis=0)).ravel()
    mean_nontissue = np.asarray(data[mask_non].mean(axis=0)).ravel()

    # 5. Identify features to remove
    to_remove = mean_tissue < foldchange * mean_nontissue

    # 6. Print removed m/z names
    mz_to_remove = adata.var_names[to_remove]
    print(f"Dropping {len(mz_to_remove)} m/z features:", mz_to_remove.values)

    # 7. Subset and return
    return adata[:, ~to_remove].copy()

In [ ]:
foldchange = 3

# Filter m/z features based on tissue mask
adata = filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=foldchange, layer=None, normalize_tic=False)

Dropping 662 m/z features: ['136.0150' '137.0251' '138.0287' '139.0359' '154.0255' '155.0350'
 '156.0424' '159.0070' '160.0107' '163.0405' '165.0200' '172.0540'
 '173.0578' '176.0111' '177.0146' '178.0269' '179.0362' '181.0163'
 '187.0399' '189.0757' '190.0676' '191.0379' '192.9801' '195.0905'
 '196.0981' '196.9856' '197.1082' '197.9919' '199.0008' '199.0439'
 '199.9998' '200.0492' '201.0571' '202.0613' '202.1793' '203.0369'
 '203.0680' '203.1800' '204.0398' '210.9897' '211.0595' '213.0560'
 '213.9619' '214.0577' '214.9722' '215.0298' '216.0426' '217.0514'
 '218.0560' '219.0630' '219.9751' '220.0723' '220.9800' '226.1111'
 '227.0378' '228.0455' '229.0025' '229.0488' '230.0543' '231.0686'
 '232.0653' '233.0507' '233.0774' '235.9458' '239.0343' '240.0412'
 '241.0502' '242.0613' '243.0269' '243.0677' '244.0353' '245.0458'
 '246.0516' '247.0594' '248.0624' '249.0743' '251.0421' '254.0257'
 '255.0287' '256.0338' '257.0478' '258.0498' '259.0608' '259.2436'
 '260.0386' '261.0393' '262.0491' '

In [ ]:
adata

AnnData object with n_obs × n_vars = 16605 × 338
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue', 'tic'
    var: 'mzs'
    layers: 'raw'

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from tqdm import tqdm


def compute_center_of_mass(x_coords, y_coords, intensities):
    total_intensity = np.sum(intensities)
    if total_intensity == 0:
        return np.nan, np.nan
    x_center = np.sum(x_coords * intensities) / total_intensity
    y_center = np.sum(y_coords * intensities) / total_intensity
    return x_center, y_center


def analyze_metrics(adata, export_border_points=False):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    all_border_data = []
    rows = []

    tissue_centroid_x = adata.obs.loc[adata.obs["tissue"] == True, "x"].mean()
    tissue_centroid_y = adata.obs.loc[adata.obs["tissue"] == True, "y"].mean()

    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}

        # Global center of mass and distance to tissue centroid
        all_x = adata.obs["x"].values
        all_y = adata.obs["y"].values
        all_intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index].flatten()
        x_com, y_com = compute_center_of_mass(all_x, all_y, all_intensities)

        if not np.isnan(x_com) and not np.isnan(y_com):
            dist_to_tissue_centroid = np.sqrt((x_com - tissue_centroid_x) ** 2 + (y_com - tissue_centroid_y) ** 2)
        else:
            dist_to_tissue_centroid = np.nan

        row["com_distance_to_tissue"] = dist_to_tissue_centroid

        # Background distance to tissue border for nonzero-intensity pixels
        bg_subset = adata[adata.obs["tissue"] == False]
        tissue_subset = adata[adata.obs["tissue"] == True]

        bg_x = bg_subset.obs["x"].values
        bg_y = bg_subset.obs["y"].values
        bg_intensities = bg_subset.X[:, mz_index].toarray().flatten() if hasattr(bg_subset.X, "toarray") else bg_subset.X[:, mz_index].flatten()

        tissue_x = tissue_subset.obs["x"].values
        tissue_y = tissue_subset.obs["y"].values

        if len(bg_intensities) > 0 and len(tissue_x) > 0:
            valid_mask = bg_intensities > 0
            valid_bg_coords = np.vstack([bg_x[valid_mask], bg_y[valid_mask]]).T
            tissue_coords = np.vstack([tissue_x, tissue_y]).T

            if valid_bg_coords.shape[0] > 0:
                tree = cKDTree(tissue_coords)
                distances, indices = tree.query(valid_bg_coords)

                nearest_tissue_points = tissue_coords[indices]

                row["bg_dist_max"] = np.max(distances)
                row["bg_dist_mean"] = np.mean(distances)
                row["bg_nonzero_area"] = valid_bg_coords.shape[0]

                if export_border_points:
                    for i in range(len(valid_bg_coords)):
                        all_border_data.append({
                            "mz": mz_values[mz_index],
                            "bg_x": valid_bg_coords[i, 0],
                            "bg_y": valid_bg_coords[i, 1],
                            "tissue_x": nearest_tissue_points[i, 0],
                            "tissue_y": nearest_tissue_points[i, 1],
                            "distance": distances[i]
                        })

            else:
                row["bg_dist_max"] = np.nan
                row["bg_dist_mean"] = np.nan
                row["bg_nonzero_area"] = 0
        else:
            row["bg_dist_max"] = np.nan
            row["bg_dist_mean"] = np.nan
            row["bg_nonzero_area"] = 0

        # Weighted centroids
        tissue_intensities = tissue_subset.X[:, mz_index].toarray().flatten() if hasattr(tissue_subset.X, "toarray") else tissue_subset.X[:, mz_index].flatten()
        tissue_centroid_x_weighted, tissue_centroid_y_weighted = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)
        row["weighted_com_x_tissue"] = tissue_centroid_x_weighted
        row["weighted_com_y_tissue"] = tissue_centroid_y_weighted
        row["weighted_com_x_all"] = x_com
        row["weighted_com_y_all"] = y_com

        rows.append(row)

    df_metrics = pd.DataFrame(rows)

    if export_border_points:
        df_border = pd.DataFrame(all_border_data)
        return df_metrics, df_border
    else:
        return df_metrics

In [ ]:
pre_df_metrics, pre_df_border = analyze_metrics(adata, export_border_points=True)

Processing m/z features: 100%|██████████| 338/338 [00:07<00:00, 43.87it/s]


In [ ]:
def mask_low_background_intensities(adata, fold_change=0.2):
    """
    Set background pixel intensities (tissue==0) to 0 if they are below fold_change * max tissue intensity per m/z.
    
    Parameters:
        adata : AnnData
            An AnnData object where .X is (n_pixels x n_mz) and .obs["tissue"] exists.
        fold_change : float
            The threshold multiplier to compare background intensity to max tissue intensity.
    """
    X = adata.X
    tissue_mask = adata.obs["tissue"] == 1
    background_mask = ~tissue_mask

    if sp.issparse(X):
        X = X.tocsr()

    # Compute max intensity of each m/z in tissue pixels
    max_tissue_intensity = X[tissue_mask.values].max(axis=0).A1 if sp.issparse(X) else X[tissue_mask.values].max(axis=0)

    # Define threshold per m/z
    thresholds = fold_change * max_tissue_intensity

    # Modify background pixels
    bg_indices = np.where(background_mask)[0]

    for i in bg_indices:
        row = X[i]
        if sp.issparse(X):
            row = row.tocoo()
            mask = row.data <= thresholds[row.col]
            row.data[mask] = 0
            X[i] = sp.csr_matrix((row.data, (np.zeros_like(row.col), row.col)), shape=(1, X.shape[1]))
        else:
            mask = row <= thresholds
            row[mask] = 0
            X[i] = row

    adata.X = X
    return adata

In [ ]:
min_background_intensity = 0.1
adata = mask_low_background_intensities(adata, fold_change=min_background_intensity)

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from tqdm import tqdm


def compute_center_of_mass(x_coords, y_coords, intensities):
    total_intensity = np.sum(intensities)
    if total_intensity == 0:
        return np.nan, np.nan
    x_center = np.sum(x_coords * intensities) / total_intensity
    y_center = np.sum(y_coords * intensities) / total_intensity
    return x_center, y_center


def analyze_metrics(adata, export_border_points=False):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    all_border_data = []
    rows = []

    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):

        # Initialize row for current m/z
        row = {"mz": mz_values[mz_index]}

        # Extract x, y, and intensity for current mz
        all_x = adata.obs["x"].values
        all_y = adata.obs["y"].values
        all_intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index].flatten()

        # Compute global center of mass
        x_com, y_com = compute_center_of_mass(all_x, all_y, all_intensities)

        # Only tissue pixels
        tissue_mask = adata.obs["tissue"].values == True
        tissue_x = all_x[tissue_mask]
        tissue_y = all_y[tissue_mask]
        tissue_intensities = all_intensities[tissue_mask]

        # Compute weighted tissue centroid (center of mass within tissue)
        x_com_tissue, y_com_tissue = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)

        # Compute distance from global COM to tissue COM
        if not np.isnan(x_com) and not np.isnan(y_com):
            dist_to_tissue_centroid = np.sqrt((x_com - x_com_tissue) ** 2 + (y_com - y_com_tissue) ** 2)
        else:
            dist_to_tissue_centroid = np.nan

        row["com_distance_to_tissue"] = dist_to_tissue_centroid

        # Background distance to tissue border for nonzero-intensity pixels
        bg_subset = adata[adata.obs["tissue"] == False]
        tissue_subset = adata[adata.obs["tissue"] == True]

        bg_x = bg_subset.obs["x"].values
        bg_y = bg_subset.obs["y"].values
        bg_intensities = bg_subset.X[:, mz_index].toarray().flatten() if hasattr(bg_subset.X, "toarray") else bg_subset.X[:, mz_index].flatten()

        tissue_x = tissue_subset.obs["x"].values
        tissue_y = tissue_subset.obs["y"].values

        if len(bg_intensities) > 0 and len(tissue_x) > 0:
            valid_mask = bg_intensities > 0
            valid_bg_coords = np.vstack([bg_x[valid_mask], bg_y[valid_mask]]).T
            tissue_coords = np.vstack([tissue_x, tissue_y]).T

            if valid_bg_coords.shape[0] > 0:
                tree = cKDTree(tissue_coords)
                distances, indices = tree.query(valid_bg_coords)

                nearest_tissue_points = tissue_coords[indices]

                row["bg_dist_max"] = np.max(distances)
                row["bg_dist_mean"] = np.mean(distances)
                row["bg_nonzero_area"] = valid_bg_coords.shape[0]

                if export_border_points:
                    for i in range(len(valid_bg_coords)):
                        all_border_data.append({
                            "mz": mz_values[mz_index],
                            "bg_x": valid_bg_coords[i, 0],
                            "bg_y": valid_bg_coords[i, 1],
                            "tissue_x": nearest_tissue_points[i, 0],
                            "tissue_y": nearest_tissue_points[i, 1],
                            "distance": distances[i]
                        })

            else:
                row["bg_dist_max"] = np.nan
                row["bg_dist_mean"] = np.nan
                row["bg_nonzero_area"] = 0
        else:
            row["bg_dist_max"] = np.nan
            row["bg_dist_mean"] = np.nan
            row["bg_nonzero_area"] = 0

        # Weighted centroids
        tissue_intensities = tissue_subset.X[:, mz_index].toarray().flatten() if hasattr(tissue_subset.X, "toarray") else tissue_subset.X[:, mz_index].flatten()
        tissue_centroid_x_weighted, tissue_centroid_y_weighted = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)
        row["weighted_com_x_tissue"] = tissue_centroid_x_weighted
        row["weighted_com_y_tissue"] = tissue_centroid_y_weighted
        row["weighted_com_x_all"] = x_com
        row["weighted_com_y_all"] = y_com

        rows.append(row)

    df_metrics = pd.DataFrame(rows)

    if export_border_points:
        df_border = pd.DataFrame(all_border_data)
        return df_metrics, df_border
    else:
        return df_metrics

In [ ]:
def visualize_distance_map(df_border, mz_value, n_arrows=100):
    subset = df_border[df_border["mz"] == mz_value]
    if subset.empty:
        print(f"No border data found for m/z {mz_value}")
        return

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(subset["bg_x"], subset["bg_y"], c="blue", s=10, label="Background Pixels")
    ax.scatter(subset["tissue_x"], subset["tissue_y"], c="red", s=10, label="Nearest Tissue Border")

    # Draw arrows from bg pixel to nearest tissue
    sample = subset.sample(n=min(n_arrows, len(subset)))
    for _, row in sample.iterrows():
        ax.arrow(row["bg_x"], row["bg_y"],
                 row["tissue_x"] - row["bg_x"],
                 row["tissue_y"] - row["bg_y"],
                 color="gray", alpha=0.5, head_width=1)

    ax.set_title(f"Nearest Tissue Border Points for m/z {mz_value}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    ax.axis("equal")
    plt.tight_layout()
    plt.show()

In [ ]:
df_metrics, df_border = analyze_metrics(adata, export_border_points=True)

Processing m/z features: 100%|██████████| 338/338 [00:04<00:00, 73.72it/s]


In [ ]:
# Visualize arrows for m/z=885.55 (example)
visualize_distance_map(df_border, mz_value=772.5231)


In [ ]:
from skimage.measure import find_contours
import matplotlib.pyplot as plt
import numpy as np

def visualize_mz_feature(adata, df_metrics, df_border, mz_value, cmap="hot"):
    # Find index of target m/z
    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))

    # Get coordinates and intensities
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    intensities = adata.X[:, mz_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_idx].flatten()

    tissue_mask = adata.obs["tissue"].values

    # Extract COMs and border points
    mz_metrics = df_metrics[df_metrics["mz"] == mz_value].iloc[0]
    com_all = (mz_metrics["weighted_com_x_all"], mz_metrics["weighted_com_y_all"])
    com_tissue = (mz_metrics["weighted_com_x_tissue"], mz_metrics["weighted_com_y_tissue"])

    df_mz_border = df_border[df_border["mz"] == mz_value]
    max_row = df_mz_border.loc[df_mz_border["distance"].idxmax()]
    bg_point = (max_row["bg_x"], max_row["bg_y"])
    nearest_tissue_point = (max_row["tissue_x"], max_row["tissue_y"])

    # Create 2D grid (image) for tissue mask
    grid_x = np.unique(x)
    grid_y = np.unique(y)
    x_to_idx = {v: i for i, v in enumerate(grid_x)}
    y_to_idx = {v: i for i, v in enumerate(grid_y)}
    mask_2d = np.zeros((len(grid_y), len(grid_x)), dtype=bool)

    for xi, yi, is_tissue in zip(x, y, tissue_mask):
        if is_tissue:
            mask_2d[y_to_idx[yi], x_to_idx[xi]] = True

    contours = find_contours(mask_2d.astype(float), level=0.5)

    # Create figure
    plt.figure(figsize=(10, 8))

    # Plot heatmap with square pixels
    plt.scatter(x, y, c=intensities, s=8, marker='s', cmap=cmap)
    plt.colorbar(label=f"Intensity @ m/z {mz_value:.2f}")

    # Plot center of mass
    plt.scatter(*com_all, color="cyan", s=150, marker="*", label="COM (all)")
    plt.scatter(*com_tissue, color="lime", s=150, marker="*", label="COM (tissue)")

    # Max distance point and nearest tissue point
    plt.plot([bg_point[0], nearest_tissue_point[0]], [bg_point[1], nearest_tissue_point[1]], 
             color="orange", linestyle="--", label="Max dist to tissue")
    plt.scatter(*bg_point, color="red", s=80, edgecolor="black", label="Furthest BG pt")
    plt.scatter(*nearest_tissue_point, color="green", s=80, edgecolor="black", label="Nearest tissue pt")

    # Plot tissue margin line
    for contour in contours:
        contour_x = grid_x[(contour[:, 1]).astype(int)]
        contour_y = grid_y[(contour[:, 0]).astype(int)]
        plt.plot(contour_x, contour_y, color="red", linewidth=2, label="Tissue border")

    plt.legend()
    plt.title(f"m/z {mz_value:.2f} Heatmap with COMs and Furthest BG Point")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.gca().set_aspect('equal')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),   # white
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])
#custom_cmap = "binary"
visualize_mz_feature(adata, df_metrics, df_border, mz_value=772.5231, cmap=custom_cmap)

In [ ]:
def visualize_mz_feature(adata, df_metrics, df_border, mz_value, cmap="hot", save_path=None):
    from skimage.measure import find_contours
    import matplotlib.pyplot as plt
    import numpy as np

    # Find index of target m/z
    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))

    # Get coordinates and intensities
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    intensities = adata.X[:, mz_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_idx].flatten()

    tissue_mask = adata.obs["tissue"].values

    # Extract COMs and border points
    mz_metrics = df_metrics[df_metrics["mz"] == mz_value].iloc[0]
    com_all = (mz_metrics["weighted_com_x_all"], mz_metrics["weighted_com_y_all"])
    com_tissue = (mz_metrics["weighted_com_x_tissue"], mz_metrics["weighted_com_y_tissue"])

    df_mz_border = df_border[df_border["mz"] == mz_value]
    if df_mz_border.empty:
        print(f"⚠️ Skipping m/z {mz_value:.4f} — no matching metric data found.")
    else:
        max_row = df_mz_border.loc[df_mz_border["distance"].idxmax()]
        bg_point = (max_row["bg_x"], max_row["bg_y"])
        nearest_tissue_point = (max_row["tissue_x"], max_row["tissue_y"])

    # Create 2D grid (image) for tissue mask
    grid_x = np.unique(x)
    grid_y = np.unique(y)
    x_to_idx = {v: i for i, v in enumerate(grid_x)}
    y_to_idx = {v: i for i, v in enumerate(grid_y)}
    mask_2d = np.zeros((len(grid_y), len(grid_x)), dtype=bool)

    for xi, yi, is_tissue in zip(x, y, tissue_mask):
        if is_tissue:
            mask_2d[y_to_idx[yi], x_to_idx[xi]] = True

    contours = find_contours(mask_2d.astype(float), level=0.5)

    # Create figure
    plt.figure(figsize=(10, 8))

    # Plot heatmap
    plt.scatter(x, y, c=intensities, s=8, marker='s', cmap=cmap)
    plt.colorbar(label=f"Intensity @ m/z {mz_value:.2f}")

    # Plot COMs and max distance markers
    plt.scatter(*com_all, color="cyan", s=150, marker="*", label="COM (all)")
    plt.scatter(*com_tissue, color="lime", s=150, marker="*", label="COM (tissue)")
    if not df_mz_border.empty:
        plt.plot([bg_point[0], nearest_tissue_point[0]], [bg_point[1], nearest_tissue_point[1]], 
                color="orange", linestyle="--", label="Max dist to tissue")
        plt.scatter(*bg_point, color="red", s=80, edgecolor="black", label="Furthest BG pt")
        plt.scatter(*nearest_tissue_point, color="green", s=80, edgecolor="black", label="Nearest tissue pt")

    # Plot tissue contour
    for contour in contours:
        contour_x = grid_x[(contour[:, 1]).astype(int)]
        contour_y = grid_y[(contour[:, 0]).astype(int)]
        plt.plot(contour_x, contour_y, color="red", linewidth=2, label="Tissue border")

    plt.legend()
    plt.title(f"m/z {mz_value:.2f} Heatmap with COMs and Furthest BG Point")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.gca().set_aspect('equal')
    plt.gca().invert_yaxis()
    plt.tight_layout()

    # Save or show
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()


In [ ]:
df_metrics

,mz,com_distance_to_tissue,bg_dist_max,bg_dist_mean,bg_nonzero_area,weighted_com_x_tissue,weighted_com_y_tissue,weighted_com_x_all,weighted_com_y_all
0,136.0608,1.193621,19.646883,3.702903,1396,64.053062,50.571565,63.782108,49.409104
1,161.1338,4.100488,39.458839,10.396888,1783,64.037348,57.038179,64.484212,61.114245
2,180.0717,5.331490,49.193496,12.060899,6081,63.922219,55.550936,64.479618,60.853208
3,184.0720,0.340549,24.351591,3.770018,1508,63.977132,54.278261,64.199891,54.535850
4,185.0686,0.594481,49.193496,9.423498,3410,64.286669,53.772173,64.847637,53.575392
...,...,...,...,...,...,...,...,...,...
333,1520.1588,1.330614,18.027756,2.181553,582,64.146848,53.183229,64.421007,54.485293
334,1521.1633,1.378289,24.083189,2.017429,496,64.054169,53.506352,64.519920,54.803563
335,1522.1682,1.212370,25.000000,2.287575,536,64.066254,53.840359,64.405934,55.004170
336,1542.1475,0.817690,23.706539,1.937516,395,63.803247,52.216174,63.796846,53.033838


In [ ]:
def score_mz_features(df_metrics, area_weight=0.95):
    # Normalize metrics
    # Normalize distance metrics
    df_metrics["normalized_bg_dist_mean"] = (
        df_metrics["bg_dist_mean"] - df_metrics["bg_dist_mean"].min()
    ) / (df_metrics["bg_dist_mean"].max() - df_metrics["bg_dist_mean"].min())

    # Normalize area metrics
    df_metrics["normalized_bg_nonzero_area"] = (
        df_metrics["bg_nonzero_area"] - df_metrics["bg_nonzero_area"].min()
    ) / (df_metrics["bg_nonzero_area"].max() - df_metrics["bg_nonzero_area"].min())

    # Calculate dispersion score
    # The dispersion score is a weighted combination of the normalized metrics
    # Higher score means better dispersion (lower distance, higher area)
    # Note: area_weight controls the balance between area and distance
    df_metrics["dispersion_score"] = (
        area_weight * df_metrics["normalized_bg_nonzero_area"]
        + (1 - area_weight) * df_metrics["normalized_bg_dist_mean"]
    )
    
    return df_metrics


In [43]:
area_weight = 0.95
# Score m/z features based on dispersion metrics
# This will add a new column 'dispersion_score' to df_metrics
df_metrics = score_mz_features(df_metrics, area_weight=area_weight)
df_metrics

,mz,com_distance_to_tissue,bg_dist_max,bg_dist_mean,bg_nonzero_area,weighted_com_x_tissue,weighted_com_y_tissue,weighted_com_x_all,weighted_com_y_all,normalized_bg_dist_mean,normalized_bg_nonzero_area,dispersion_score
0,136.0608,1.193621,19.646883,3.702903,1396,64.053062,50.571565,63.782108,49.409104,0.202837,0.220077,0.219215
1,161.1338,4.100488,39.458839,10.396888,1783,64.037348,57.038179,64.484212,61.114245,0.754486,0.284501,0.308001
2,180.0717,5.331490,49.193496,12.060899,6081,63.922219,55.550936,64.479618,60.853208,0.891617,1.000000,0.994581
3,184.0720,0.340549,24.351591,3.770018,1508,63.977132,54.278261,64.199891,54.535850,0.208368,0.238721,0.237204
4,185.0686,0.594481,49.193496,9.423498,3410,64.286669,53.772173,64.847637,53.575392,0.674270,0.555352,0.561298
...,...,...,...,...,...,...,...,...,...,...,...,...
333,1520.1588,1.330614,18.027756,2.181553,582,64.146848,53.183229,64.421007,54.485293,0.077464,0.084568,0.084213
334,1521.1633,1.378289,24.083189,2.017429,496,64.054169,53.506352,64.519920,54.803563,0.063938,0.070251,0.069936
335,1522.1682,1.212370,25.000000,2.287575,536,64.066254,53.840359,64.405934,55.004170,0.086201,0.076910,0.077375
336,1542.1475,0.817690,23.706539,1.937516,395,63.803247,52.216174,63.796846,53.033838,0.057353,0.053438,0.053633


In [44]:
df_metrics.to_csv("mz_metrics.csv", index=False)

In [ ]:
output_folder = f"output_{min_background_intensity}_score_{area_weight}"

if os.path.exists(output_folder):
    print(f"⚠️ Warning: Output folder '{output_folder}' already exists. Overwriting.")
else:
    os.makedirs(output_folder, exist_ok=True)

df_sorted = df_metrics.sort_values(by="dispersion_score", ascending=False)
rank_num = 1

# Visualize and save m/z features based on dispersion score
# Loop through sorted m/z values and save visualizations
for mz in df_sorted["mz"]:
    filename = f"{rank_num}_mz_{mz:.4f}.png"
    filepath = os.path.join(output_folder, filename)
    
    custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
        (0.0, "#454545"),   # white
        (0.00000001, "#000000"),  # black
        (0.10, "#000080"),  # navy
        (0.15, "#0000FF"),  # blue
        (0.30, "#8000FF"),  # purple-ish
        (0.45, "#FF0000"),  # red
        (0.60, "#FF8000"),  # orange
        (0.75, "#FFFF00"),  # yellow
        (1.0, "#FFFFFF")   # white
    ])
    
    visualize_mz_feature(adata, df_metrics, df_border, mz_value=mz, cmap=custom_cmap, save_path=filepath)
    print(f"Saved: {filepath}")
    rank_num += 1

⚠️ Warning: Output folder 'output_0.1_score_0.95' already exists. Overwriting.
Saved: output_0.1_score_0.95/1_mz_180.0717.png
Saved: output_0.1_score_0.95/2_mz_370.3516.png
Saved: output_0.1_score_0.95/3_mz_367.3328.png
Saved: output_0.1_score_0.95/4_mz_371.3606.png
Saved: output_0.1_score_0.95/5_mz_369.3523.png
Saved: output_0.1_score_0.95/6_mz_284.0794.png
Saved: output_0.1_score_0.95/7_mz_368.3461.png
Saved: output_0.1_score_0.95/8_mz_341.3084.png
Saved: output_0.1_score_0.95/9_mz_385.3478.png
Saved: output_0.1_score_0.95/10_mz_185.0686.png
Saved: output_0.1_score_0.95/11_mz_243.2107.png
Saved: output_0.1_score_0.95/12_mz_328.3184.png
Saved: output_0.1_score_0.95/13_mz_365.3188.png
Saved: output_0.1_score_0.95/14_mz_383.3277.png
Saved: output_0.1_score_0.95/15_mz_808.5768.png
Saved: output_0.1_score_0.95/16_mz_386.3513.png
Saved: output_0.1_score_0.95/17_mz_735.5735.png
Saved: output_0.1_score_0.95/18_mz_384.3286.png
Saved: output_0.1_score_0.95/19_mz_366.3293.png
Saved: output_0.1_

In [ ]:
"""import pandas as pd

# Replace 'your_file.csv' with the path to your CSV file
file_path = '0.1_mean_bg_dis_area.csv'

# Read the CSV file into a DataFrame
df = pd.read_csv(file_path)

# Display the first few rows of the DataFrame
print(df.head())"""

"import pandas as pd\n\n# Replace 'your_file.csv' with the path to your CSV file\nfile_path = '0.1_mean_bg_dis_area.csv'\n\n# Read the CSV file into a DataFrame\ndf = pd.read_csv(file_path)\n\n# Display the first few rows of the DataFrame\nprint(df.head())"

In [ ]:
# Merge the dataframes on the 'm/z' column
df_metrics = df_metrics.merge(df, on='mz', how='left')

# Display the updated DataFrame
print(df_metrics.head())

NameError: name 'df' is not defined

In [ ]:
'''import os
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

sorting_list = ['bg_nonzero_area', 'com_distance_to_tissue', 'bg_dist_max', 'bg_dist_mean']

for sort in sorting_list:
    output_folder = f"{min_background_intensity}_output_{sort}"

    if os.path.exists(output_folder):
        print(f"⚠️ Warning: Output folder '{output_folder}' already exists. Overwriting.")
    else:
        os.makedirs(output_folder, exist_ok=True)

    df_sorted = df_metrics.sort_values(by=sort, ascending=False)
    rank_num = 1

    output_filename = f"df_metrics_sorted_{min_background_intensity}_{sort}.csv"
    df_sorted.to_csv(output_filename, index=False, float_format="%.6f")
    print(f"✅ Saved: {output_filename}")

    for mz in df_sorted["mz"]:
        filename = f"{rank_num}_mz_{mz:.4f}.png"
        filepath = os.path.join(output_folder, filename)
        
        custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
            (0.0, "#454545"),   # white
            (0.00000001, "#000000"),  # black
            (0.10, "#000080"),  # navy
            (0.15, "#0000FF"),  # blue
            (0.30, "#8000FF"),  # purple-ish
            (0.45, "#FF0000"),  # red
            (0.60, "#FF8000"),  # orange
            (0.75, "#FFFF00"),  # yellow
            (1.0, "#FFFFFF")   # white
        ])
        
        visualize_mz_feature(adata, df_metrics, df_border, mz_value=mz, cmap=custom_cmap, save_path=filepath)
        print(f"Saved: {filepath}")
        rank_num += 1'''

✅ Saved: df_metrics_sorted_0.4_bg_nonzero_area.csv
Saved: 0.4_output_bg_nonzero_area/1_mz_808.5768.png
Saved: 0.4_output_bg_nonzero_area/2_mz_427.9302.png
Saved: 0.4_output_bg_nonzero_area/3_mz_580.9368.png
Saved: 0.4_output_bg_nonzero_area/4_mz_406.9501.png
Saved: 0.4_output_bg_nonzero_area/5_mz_734.5673.png
Saved: 0.4_output_bg_nonzero_area/6_mz_428.9334.png
Saved: 0.4_output_bg_nonzero_area/7_mz_735.5735.png
Saved: 0.4_output_bg_nonzero_area/8_mz_762.5918.png
Saved: 0.4_output_bg_nonzero_area/9_mz_774.9400.png
Saved: 0.4_output_bg_nonzero_area/10_mz_760.5916.png
Saved: 0.4_output_bg_nonzero_area/11_mz_736.5804.png
Saved: 0.4_output_bg_nonzero_area/12_mz_404.9267.png
Saved: 0.4_output_bg_nonzero_area/13_mz_136.0608.png
Saved: 0.4_output_bg_nonzero_area/14_mz_761.5914.png
Saved: 0.4_output_bg_nonzero_area/15_mz_651.5335.png
Saved: 0.4_output_bg_nonzero_area/16_mz_809.5828.png
Saved: 0.4_output_bg_nonzero_area/17_mz_385.2707.png
Saved: 0.4_output_bg_nonzero_area/18_mz_731.6000.png
Save

In [ ]:
'''from skimage.measure import find_contours
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

def visualize_mz_feature_with_he(
    adata, df_metrics, df_border, mz_value, he_image_path, pixel_size=1.0, origin=(0, 0), cmap="hot"
):
    # Load H&E image
    he_img = mpimg.imread(he_image_path)

    # Find index of target m/z
    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))

    # Get coordinates and intensities
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    intensities = adata.X[:, mz_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_idx].flatten()

    tissue_mask = adata.obs["tissue"].values

    # Extract COMs and border points
    mz_metrics = df_metrics[df_metrics["mz"] == mz_value].iloc[0]
    com_all = (mz_metrics["weighted_com_x_all"], mz_metrics["weighted_com_y_all"])
    com_tissue = (mz_metrics["weighted_com_x_tissue"], mz_metrics["weighted_com_y_tissue"])

    df_mz_border = df_border[df_border["mz"] == mz_value]
    max_row = df_mz_border.loc[df_mz_border["distance"].idxmax()]
    bg_point = (max_row["bg_x"], max_row["bg_y"])
    nearest_tissue_point = (max_row["tissue_x"], max_row["tissue_y"])

    # Convert coordinates to image pixel indices
    x_idx = ((x - origin[0]) / pixel_size).astype(int)
    y_idx = ((y - origin[1]) / pixel_size).astype(int)
    com_all_idx = [(com_all[0] - origin[0]) / pixel_size, (com_all[1] - origin[1]) / pixel_size]
    com_tissue_idx = [(com_tissue[0] - origin[0]) / pixel_size, (com_tissue[1] - origin[1]) / pixel_size]
    bg_point_idx = [(bg_point[0] - origin[0]) / pixel_size, (bg_point[1] - origin[1]) / pixel_size]
    nearest_tissue_point_idx = [(nearest_tissue_point[0] - origin[0]) / pixel_size, (nearest_tissue_point[1] - origin[1]) / pixel_size]

    # Create 2D tissue mask image
    grid_x = np.unique(x_idx)
    grid_y = np.unique(y_idx)
    x_to_i = {v: i for i, v in enumerate(grid_x)}
    y_to_i = {v: i for i, v in enumerate(grid_y)}
    mask_2d = np.zeros((len(grid_y), len(grid_x)), dtype=bool)

    for xi, yi, is_tissue in zip(x_idx, y_idx, tissue_mask):
        if is_tissue and xi in x_to_i and yi in y_to_i:
            mask_2d[y_to_i[yi], x_to_i[xi]] = True

    contours = find_contours(mask_2d.astype(float), level=0.5)

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(he_img)

    # Overlay MALDI heatmap
    sc = ax.scatter(x_idx, y_idx, c=intensities, s=100, cmap=cmap, marker='s', alpha=0.6)

    # Overlay COMs
    ax.scatter(*com_all_idx, color="cyan", s=150, marker="*", label="COM (all)")
    ax.scatter(*com_tissue_idx, color="blue", s=150, marker="*", label="COM (tissue)")

    # Overlay furthest point
    ax.plot([bg_point_idx[0], nearest_tissue_point_idx[0]], [bg_point_idx[1], nearest_tissue_point_idx[1]],
             color="orange", linestyle="--", label="Max dist to tissue")
    ax.scatter(*bg_point_idx, color="red", s=80, edgecolor="black", label="Furthest BG pt")
    ax.scatter(*nearest_tissue_point_idx, color="green", s=80, edgecolor="black", label="Nearest tissue pt")

    # Overlay tissue margin
    for contour in contours:
        contour_x = grid_x[(contour[:, 1]).astype(int)]
        contour_y = grid_y[(contour[:, 0]).astype(int)]
        ax.plot(contour_x, contour_y, color="white", linewidth=1.5, label="Tissue border")

    ax.set_title(f"Overlay @ m/z {mz_value:.2f}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.invert_yaxis()
    ax.set_aspect('equal')
    plt.colorbar(sc, ax=ax, label=f"Intensity @ m/z {mz_value:.2f}")
    ax.legend()
    plt.tight_layout()
    plt.show()
'''

'from skimage.measure import find_contours\nimport matplotlib.pyplot as plt\nimport matplotlib.image as mpimg\nimport numpy as np\n\ndef visualize_mz_feature_with_he(\n    adata, df_metrics, df_border, mz_value, he_image_path, pixel_size=1.0, origin=(0, 0), cmap="hot"\n):\n    # Load H&E image\n    he_img = mpimg.imread(he_image_path)\n\n    # Find index of target m/z\n    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))\n\n    # Get coordinates and intensities\n    x = adata.obs["x"].values\n    y = adata.obs["y"].values\n    intensities = adata.X[:, mz_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_idx].flatten()\n\n    tissue_mask = adata.obs["tissue"].values\n\n    # Extract COMs and border points\n    mz_metrics = df_metrics[df_metrics["mz"] == mz_value].iloc[0]\n    com_all = (mz_metrics["weighted_com_x_all"], mz_metrics["weighted_com_y_all"])\n    com_tissue = (mz_metrics["weighted_com_x_tissue"], mz_metrics["weighted_com_y_tissue"])\n\n

In [ ]:
'''visualize_mz_feature_with_he(
    adata,
    df_metrics,
    df_border,
    mz_value=772.5231,
    he_image_path="path/to/your/he_image.png",
    pixel_size=10.0,  # adjust to match your spatial scale
    origin=(0, 0),
    cmap="hot"
)'''

'visualize_mz_feature_with_he(\n    adata,\n    df_metrics,\n    df_border,\n    mz_value=772.5231,\n    he_image_path="path/to/your/he_image.png",\n    pixel_size=10.0,  # adjust to match your spatial scale\n    origin=(0, 0),\n    cmap="hot"\n)'

In [ ]:
'''from skimage.transform import AffineTransform, warp

# Define transform manually
transform = AffineTransform(
    scale=(pixel_size, pixel_size),   # e.g., (10, 10) if MALDI is 10 µm/pixel
    translation=(-origin_x, -origin_y),  # shift MALDI to H&E
    rotation=np.deg2rad(angle)       # optional
)'''

'from skimage.transform import AffineTransform, warp\n\n# Define transform manually\ntransform = AffineTransform(\n    scale=(pixel_size, pixel_size),   # e.g., (10, 10) if MALDI is 10 µm/pixel\n    translation=(-origin_x, -origin_y),  # shift MALDI to H&E\n    rotation=np.deg2rad(angle)       # optional\n)'

In [ ]:
'''# You supply these
msi_points = np.array([[x1_msi, y1_msi], [x2_msi, y2_msi], [x3_msi, y3_msi]])
he_points = np.array([[x1_he, y1_he], [x2_he, y2_he], [x3_he, y3_he]])'''

'# You supply these\nmsi_points = np.array([[x1_msi, y1_msi], [x2_msi, y2_msi], [x3_msi, y3_msi]])\nhe_points = np.array([[x1_he, y1_he], [x2_he, y2_he], [x3_he, y3_he]])'

In [ ]:
'''from skimage.transform import estimate_transform

tform = estimate_transform('affine', msi_points, he_points)'''

"from skimage.transform import estimate_transform\n\ntform = estimate_transform('affine', msi_points, he_points)"

In [ ]:
'''transformed_coords = tform(adata.obs[["x", "y"]].values)
'''

'transformed_coords = tform(adata.obs[["x", "y"]].values)\n'

In [ ]:
'''import imreg_dft as ird
from skimage import io

# Load images
ref_img = io.imread("he_image.png", as_gray=True)
mov_img = maldi_heatmap  # e.g., summed MALDI intensity as 2D image

# Align
result = ird.similarity(ref_img, mov_img, numiter=3)
aligned_maldi = ird.transform_img(mov_img, tvec=result["tvec"])'''

'import imreg_dft as ird\nfrom skimage import io\n\n# Load images\nref_img = io.imread("he_image.png", as_gray=True)\nmov_img = maldi_heatmap  # e.g., summed MALDI intensity as 2D image\n\n# Align\nresult = ird.similarity(ref_img, mov_img, numiter=3)\naligned_maldi = ird.transform_img(mov_img, tvec=result["tvec"])'